In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Hybrid retrieval evaluation

This notebook evaluates lexical retrieval, dense embedding retrieval, and a hybrid of both methods on the 5,000-email stratified Enron sample and 250 evaluation questions. The hybrid uses reciprocal rank fusion (RRF).

In [ ]:
# Configure the E5 dense retriever and reciprocal rank fusion settings.
import gc
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

SAMPLED_EMAILS_FILE = Path("enron_stratified_sample.csv")
EVALUATION_QUESTIONS_FILE = Path("data/enron_evaluation_250.csv")

MODEL_NAME = "intfloat/e5-base-v2"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "
EMBEDDING_BATCH_SIZE = 64

FUSION_DEPTH = 100
RRF_CONSTANT = 60
LEXICAL_WEIGHT = 1.0
DENSE_WEIGHT = 1.0

In [ ]:
# Load both datasets, validate their schemas, and confirm every labelled source exists.
# The shared search_text definition keeps this notebook comparable with the other evaluations.
sampled_emails = pd.read_csv(SAMPLED_EMAILS_FILE)
evaluation_questions = pd.read_csv(EVALUATION_QUESTIONS_FILE)

required_email_columns = {"email_id", "sender", "recipients", "subject", "body"}
required_question_columns = {"question", "answer", "email_id", "difficulty", "retrieval_type"}
missing_email_columns = required_email_columns - set(sampled_emails.columns)
missing_question_columns = required_question_columns - set(evaluation_questions.columns)
if missing_email_columns or missing_question_columns:
    raise ValueError(
        f"Missing email columns: {sorted(missing_email_columns)}; "
        f"missing evaluation columns: {sorted(missing_question_columns)}"
    )

sampled_emails["email_id"] = sampled_emails["email_id"].astype(str)
evaluation_questions["email_id"] = evaluation_questions["email_id"].astype(str)
missing_sources = set(evaluation_questions["email_id"]) - set(sampled_emails["email_id"])
if missing_sources:
    raise ValueError(f"{len(missing_sources)} evaluation source emails are missing from the sample")

# Combine email metadata and body text into a single searchable representation.
def build_search_text(email_frame):
    """Build the canonical retrieval text used by every evaluation notebook."""
    return (
        email_frame["sender"].fillna("") + " " +
        email_frame["recipients"].fillna("") + " " +
        email_frame["subject"].fillna("") + " " +
        email_frame["body"].fillna("")
    )


sampled_emails["search_text"] = build_search_text(sampled_emails)

print(f"Loaded {len(sampled_emails):,} emails and {len(evaluation_questions):,} questions.")

## Lexical index

TF-IDF provides the lexical side of the hybrid. It operates at email level and is especially useful for exact names, dates, identifiers, and unusual terminology.

In [ ]:
# Build the TF-IDF retriever and rank all emails for each evaluation question.
lexical_started = time.perf_counter()
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True,
)
email_tfidf_matrix = tfidf_vectorizer.fit_transform(sampled_emails["search_text"])
question_tfidf_matrix = tfidf_vectorizer.transform(evaluation_questions["question"].astype(str))
lexical_scores = (question_tfidf_matrix @ email_tfidf_matrix.T).toarray().astype(np.float32)
lexical_order = np.argsort(lexical_scores, axis=1)[:, ::-1]
lexical_seconds = time.perf_counter() - lexical_started
print(f"Built TF-IDF index and ranked all questions in {lexical_seconds:.1f} seconds.")

## Dense index

The dense side retrieves semantically related text even when the question and email use different wording. Each email can produce several chunks, but evaluation and fusion are performed at email level.

In [ ]:
# Split emails into overlapping E5 token chunks and retain their parent email positions.
def create_chunks(emails, embedding_model):
    if CHUNK_OVERLAP >= CHUNK_SIZE:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")
    if CHUNK_SIZE >= embedding_model.max_seq_length:
        raise ValueError("CHUNK_SIZE must leave room for prefixes and special tokens")

    tokenizer = embedding_model.tokenizer
    step = CHUNK_SIZE - CHUNK_OVERLAP
    records = []

    for email_position, email in enumerate(emails.itertuples(index=False)):
        token_ids = tokenizer.encode(
            str(email.search_text),
            add_special_tokens=False,
            truncation=False,
        )

        for chunk_number, start in enumerate(range(0, len(token_ids), step)):
            end = min(start + CHUNK_SIZE, len(token_ids))
            chunk_token_ids = token_ids[start:end]

            if not chunk_token_ids:
                continue

            chunk_text = tokenizer.decode(
                chunk_token_ids,
                skip_special_tokens=True,
            )

            records.append({
                "email_position": email_position,
                "email_id": email.email_id,
                "chunk_number": chunk_number,
                "chunk_text": chunk_text,
                "embedding_text": PASSAGE_PREFIX + chunk_text,
            })

            if end == len(token_ids):
                break

    return pd.DataFrame(records)

In [ ]:
# Build the E5 dense retriever and retain each email's highest scoring chunk.
dense_started = time.perf_counter()

embedding_model = SentenceTransformer(MODEL_NAME)
chunks = create_chunks(sampled_emails, embedding_model)

chunk_embeddings = embedding_model.encode(
    chunks["embedding_text"].tolist(),
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

prefixed_questions = [
    QUERY_PREFIX + str(question)
    for question in evaluation_questions["question"]
]

question_embeddings = embedding_model.encode(
    prefixed_questions,
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype(np.float32)

chunk_scores = question_embeddings @ chunk_embeddings.T
chunk_email_positions = chunks["email_position"].to_numpy(dtype=np.int64)

dense_scores = np.full(
    (len(evaluation_questions), len(sampled_emails)),
    -np.inf,
    dtype=np.float32,
)

for question_position, scores in enumerate(chunk_scores):
    np.maximum.at(
        dense_scores[question_position],
        chunk_email_positions,
        scores,
    )

dense_order = np.argsort(dense_scores, axis=1)[:, ::-1]

dense_seconds = time.perf_counter() - dense_started

print(
    f"Built {len(chunks):,} chunks with {MODEL_NAME} "
    f"and ranked all questions in {dense_seconds:.1f} seconds."
)

## Reciprocal rank fusion

Only the top `FUSION_DEPTH` results from each retriever contribute to the hybrid score. Equal weights are a sensible untuned baseline. If weights are later tuned, use a separate development set rather than the final test questions.

In [ ]:
# Fuse the top lexical and dense rankings with Reciprocal Rank Fusion.
# RRF combines rank positions rather than incompatible TF-IDF and embedding scores.
def reciprocal_rank_fusion_scores(lexical_ranking, dense_ranking):
    fused_scores = np.zeros(len(sampled_emails), dtype=np.float32)
    depth = min(FUSION_DEPTH, len(sampled_emails))
    rank_numbers = np.arange(1, depth + 1, dtype=np.float32)
    fused_scores[lexical_ranking[:depth]] += (
        LEXICAL_WEIGHT / (RRF_CONSTANT + rank_numbers)
    )
    fused_scores[dense_ranking[:depth]] += (
        DENSE_WEIGHT / (RRF_CONSTANT + rank_numbers)
    )
    return fused_scores

hybrid_scores = np.vstack([
    reciprocal_rank_fusion_scores(lexical_order[i], dense_order[i])
    for i in range(len(evaluation_questions))
])
hybrid_order = np.argsort(hybrid_scores, axis=1)[:, ::-1]

print(
    f"Fused the top {FUSION_DEPTH} results from each retriever with "
    f"weights lexical={LEXICAL_WEIGHT:g}, dense={DENSE_WEIGHT:g}."
)

In [ ]:
# Calculate source email ranks and retrieval metrics for TF-IDF, E5, and hybrid retrieval.
email_ids = sampled_emails["email_id"].to_numpy()
target_ids = evaluation_questions["email_id"].to_numpy()

email_position_by_id = {
    email_id: position
    for position, email_id in enumerate(email_ids)
}

# Find the labelled source email's position within each retrieval ranking.
def relevant_ranks(ranking_matrix, candidate_scores=None):
    ranks = []

    for question_position, (ranking, target_id) in enumerate(
        zip(ranking_matrix, target_ids)
    ):
        target_position = email_position_by_id[target_id]

        if (
            candidate_scores is not None
            and candidate_scores[question_position, target_position] <= 0
        ):
            ranks.append(np.inf)
            continue

        matches = np.flatnonzero(email_ids[ranking] == target_id)
        ranks.append(int(matches[0]) + 1 if len(matches) else np.inf)

    return np.asarray(ranks, dtype=float)


# Convert source ranks into Recall@K and mean reciprocal rank metrics.
def metric_record(method, ranks):
    return {
        "method": method,
        "recall_at_1": float(np.mean(ranks <= 1)),
        "recall_at_3": float(np.mean(ranks <= 3)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "recall_at_10": float(np.mean(ranks <= 10)),
        "recall_at_50": float(np.mean(ranks <= 50)),
        "recall_at_100": float(np.mean(ranks <= 100)),
        "mrr": float(np.mean(1.0 / ranks)),
    }


method_ranks = {
    "TF-IDF": relevant_ranks(lexical_order),
    "Dense E5": relevant_ranks(dense_order),
    "Hybrid RRF": relevant_ranks(
        hybrid_order,
        candidate_scores=hybrid_scores,
    ),
}

summary_results = pd.DataFrame([
    metric_record(name, ranks)
    for name, ranks in method_ranks.items()
]).sort_values(
    ["recall_at_10", "mrr"],
    ascending=False,
).reset_index(drop=True)

In [ ]:
# Display the overall TF-IDF, E5, and hybrid retrieval results.
display(
    summary_results.style
    .format({
        "recall_at_1": "{:.1%}",
        "recall_at_3": "{:.1%}",
        "recall_at_5": "{:.1%}",
        "recall_at_10": "{:.1%}",
        "recall_at_50": "{:.1%}",
        "recall_at_100": "{:.1%}",
        "mrr": "{:.3f}",
    })
    .background_gradient(
        subset=["recall_at_1", "recall_at_3", "recall_at_5", "recall_at_10", "recall_at_50", "recall_at_100", "mrr"],
        cmap="YlGn",
    )
)

In [ ]:
# Store the TF-IDF, E5, and hybrid source ranks for every evaluation question.
query_results = evaluation_questions[
    ["question", "answer", "email_id", "difficulty", "retrieval_type"]
].copy()

query_results["tfidf_rank"] = method_ranks["TF-IDF"]
query_results["dense_rank"] = method_ranks["Dense E5"]
query_results["hybrid_rank"] = method_ranks["Hybrid RRF"]

display(query_results.head(10))

In [ ]:
# Calculate and compare retrieval performance separately for each question difficulty.
def grouped_metrics(group):
    rows = []

    for method, column in [
        ("TF-IDF", "tfidf_rank"),
        ("Dense E5", "dense_rank"),
        ("Hybrid RRF", "hybrid_rank"),
    ]:
        ranks = group[column].to_numpy(dtype=float)

        rows.append({
            "method": method,
            "questions": len(group),
            "recall_at_1": np.mean(ranks <= 1),
            "recall_at_5": np.mean(ranks <= 5),
            "recall_at_10": np.mean(ranks <= 10),
            "recall_at_50": np.mean(ranks <= 50),
            "recall_at_100": np.mean(ranks <= 100),
            "mrr": np.mean(1.0 / ranks),
        })

    return pd.DataFrame(rows)


difficulty_results = pd.concat(
    [
        grouped_metrics(group).assign(difficulty=difficulty)
        for difficulty, group in query_results.groupby(
            "difficulty",
            sort=False,
        )
    ],
    ignore_index=True,
)[
    [
        "difficulty",
        "method",
        "questions",
        "recall_at_1",
        "recall_at_5",
        "recall_at_10",
        "recall_at_50",
        "recall_at_100",
        "mrr",
    ]
]

display(
    difficulty_results.style.format({
        "recall_at_1": "{:.1%}",
        "recall_at_5": "{:.1%}",
        "recall_at_10": "{:.1%}",
        "recall_at_50": "{:.1%}",
        "recall_at_100": "{:.1%}",
        "mrr": "{:.3f}",
    })
)

In [ ]:
# Show questions brought into the top 10 by fusion when either component ranked them lower.
hybrid_rescues = query_results[
    (query_results["hybrid_rank"] <= 10)
    & ((query_results["tfidf_rank"] > 10) | (query_results["dense_rank"] > 10))
].sort_values("hybrid_rank")

print(f"Questions rescued into the hybrid top 10: {len(hybrid_rescues)}")
display(hybrid_rescues.head(20))

In [ ]:
# Plot Recall@K across TF-IDF, E5, and hybrid retrieval.
plot_data = summary_results.melt(
    id_vars="method",
    value_vars=["recall_at_1", "recall_at_3", "recall_at_5", "recall_at_10", "recall_at_50", "recall_at_100"],
    var_name="metric",
    value_name="recall",
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_data, x="metric", y="recall", hue="method")
plt.ylim(0, 1)
plt.xlabel("Metric")
plt.ylabel("Recall")
plt.title("Lexical, dense, and hybrid retrieval")
plt.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Detailed retrieval results

This table presents both the recall percentage and the number of successful questions out of the full evaluation set.

In [ ]:
# Present each Recall result as both a percentage and a question count.
evaluation_question_count = len(evaluation_questions)
retrieval_results_table = pd.DataFrame({
    "Method": summary_results["method"],
})
for cutoff in [1, 3, 5, 10, 50, 100]:
    recall_values = summary_results[f"recall_at_{cutoff}"]
    successful_questions = (
        recall_values * evaluation_question_count
    ).round().astype(int)
    retrieval_results_table[f"Recall@{cutoff}"] = [
        f"{recall:.1%} ({successes}/{evaluation_question_count})"
        for recall, successes in zip(recall_values, successful_questions)
    ]
retrieval_results_table["MRR"] = summary_results["mrr"]

display(retrieval_results_table.style.format({"MRR": "{:.3f}"}))

## Cross-encoder reranking

The first-stage hybrid retrieves 100 candidate emails for each question. `BAAI/bge-reranker-base` then reads each question and candidate passage together and reorders those candidates. The reranker cannot recover an email outside the initial top 100

In [ ]:
# Select the strongest E5 chunk from each top hybrid email for reranking.
# Build the question-passage pairs that will be scored by the BGE cross-encoder.
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
RERANK_CANDIDATES = 100
RERANKER_BATCH_SIZE = 16

chunk_indices_by_email = [[] for _ in range(len(sampled_emails))]
for chunk_index, email_position in enumerate(chunk_email_positions):
    chunk_indices_by_email[email_position].append(chunk_index)
chunk_indices_by_email = [
    np.asarray(indices, dtype=np.int64) for indices in chunk_indices_by_email
]

question_texts = evaluation_questions["question"].astype(str).to_numpy()
email_subjects = sampled_emails["subject"].fillna("").astype(str).to_numpy()
chunk_texts = chunks["chunk_text"].astype(str).to_numpy()
chunk_numbers = chunks["chunk_number"].to_numpy(dtype=np.int64)
reranker_candidate_records = []

for question_position, question in enumerate(question_texts):
    candidate_email_positions = hybrid_order[
        question_position, :RERANK_CANDIDATES
    ]
    for hybrid_rank, email_position in enumerate(
        candidate_email_positions, start=1
    ):
        candidate_chunk_indices = chunk_indices_by_email[email_position]
        if len(candidate_chunk_indices):
            best_chunk_index = candidate_chunk_indices[
                np.argmax(chunk_scores[question_position, candidate_chunk_indices])
            ]
            best_chunk_number = int(chunk_numbers[best_chunk_index])
            passage = (
                f"Subject: {email_subjects[email_position]}\n"
                f"{chunk_texts[best_chunk_index]}"
            )
        else:
            best_chunk_index = -1
            best_chunk_number = -1
            passage = f"Subject: {email_subjects[email_position]}"

        reranker_candidate_records.append({
            "question_position": question_position,
            "question": question,
            "email_position": int(email_position),
            "email_id": email_ids[email_position],
            "hybrid_rank": hybrid_rank,
            "chunk_index": int(best_chunk_index),
            "chunk_number": best_chunk_number,
            "passage": passage,
        })

reranker_candidates = pd.DataFrame(reranker_candidate_records)
print(
    f"Prepared {len(reranker_candidates):,} question-passage pairs "
    f"from the hybrid top {RERANK_CANDIDATES}."
)

In [ ]:
# Score the hybrid candidates with the BGE cross-encoder and rerank them by relevance.
# Record the new source-email rank for each evaluation question.
if torch.cuda.is_available():
    reranker_device = "cuda"
elif torch.backends.mps.is_available():
    reranker_device = "mps"
else:
    reranker_device = "cpu"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

reranker_started = time.perf_counter()
reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    max_length=512,
    device=reranker_device,
)
reranker_pairs = list(zip(
    reranker_candidates["question"],
    reranker_candidates["passage"],
))
reranker_candidates["reranker_score"] = np.asarray(
    reranker.predict(
        reranker_pairs,
        batch_size=RERANKER_BATCH_SIZE,
        show_progress_bar=True,
    )
).reshape(-1)

reranked_order = np.vstack([
    reranker_candidates[
        reranker_candidates["question_position"] == question_position
    ]
    .sort_values(
        ["reranker_score", "hybrid_rank"],
        ascending=[False, True],
    )["email_position"]
    .to_numpy(dtype=np.int64)
    for question_position in range(len(evaluation_questions))
])
reranker_ranks = relevant_ranks(reranked_order)
query_results["reranker_rank"] = reranker_ranks
reranker_seconds = time.perf_counter() - reranker_started
print(
    f"Reranked {len(reranker_candidates):,} pairs on {reranker_device} "
    f"in {reranker_seconds:.1f} seconds."
)

In [ ]:
# Compare the BGE-reranked pipeline against TF-IDF, E5, and hybrid retrieval.
reranker_method_name = "Hybrid RRF (e5) + BGE reranker"
comparison_method_ranks = {
    **method_ranks,
    reranker_method_name: reranker_ranks,
}
reranker_comparison_results = pd.DataFrame([
    metric_record(method, ranks)
    for method, ranks in comparison_method_ranks.items()
]).sort_values(["recall_at_10", "mrr"], ascending=False).reset_index(drop=True)

display(
    reranker_comparison_results.style
    .format({
        "recall_at_1": "{:.1%}",
        "recall_at_3": "{:.1%}",
        "recall_at_5": "{:.1%}",
        "recall_at_10": "{:.1%}",
        "recall_at_50": "{:.1%}",
        "recall_at_100": "{:.1%}",
        "mrr": "{:.3f}",
    })
    .background_gradient(
        subset=[
            "recall_at_1", "recall_at_3", "recall_at_5",
            "recall_at_10", "recall_at_50",
            "recall_at_100", "mrr",
        ],
        cmap="YlGn",
    )
)

In [ ]:
# Plot retrieval performance before and after BGE reranking.
reranker_plot_data = reranker_comparison_results.melt(
    id_vars="method",
    value_vars=[
        "recall_at_1", "recall_at_3", "recall_at_5",
        "recall_at_10", "recall_at_50", "recall_at_100",
    ],
    var_name="metric",
    value_name="recall",
)
plt.figure(figsize=(12, 5))
sns.barplot(
    data=reranker_plot_data,
    x="metric",
    y="recall",
    hue="method",
)
plt.ylim(0, 1)
plt.xlabel("Metric")
plt.ylabel("Recall")
plt.title("Retrieval comparison before and after reranking")
plt.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()